### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [7]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x1205e5f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1205e6990>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [14]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.9', 'langchain': '1.3.13'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x1205e5f90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1205e6990>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'

In [11]:
model.invoke("Provide details aobout the movie Inception")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - **Topic:** Movie "Inception"\n   - **Request:** "Provide details aobout the movie Inception" (Note: typo "aobout" -> "about")\n   - **Expected Output:** Comprehensive but concise details about the film, including director, release year, plot, cast, themes, awards, reception, and notable facts.\n\n2.  **Identify Key Information Needed:**\n   - Title: Inception\n   - Director: Christopher Nolan\n   - Release Year: 2010\n   - Genre: Science fiction, action, thriller, heist\n   - Cast: Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page (formerly Ellen Page), Tom Hardy, Ken Watanabe, Cillian Murphy, Tom Berenger, Marion Cotillard, Michael Caine\n   - Plot Summary: Concept of dream infiltration, dream-sharing technology, "inception" (planting an idea), main character Dom Cobb, his mission, personal stakes (Mal, children), structure of dreams within dreams\n   - Themes: Reality vs. illusion, guilt

In [15]:
response=model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output along side parsed structure

In [17]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:** The user is asking for details about the movie "Inception".\n2.  **Identify Required Information:** I need to provide details about the movie. The available tool is `Movie` which requires: `title`, `year`, `director`, `rating`.\n3.  **Gather Movie Information (Internal Knowledge):**\n   - Title: Inception\n   - Year: 2010\n   - Director: Christopher Nolan\n   - Rating: I need a rating out of 10. IMDb rating for Inception is around 8.8. I\'ll use 8.8.\n4.  **Check Tool Requirements:** The `Movie` function requires all four parameters: title, year, director, rating. All are available.\n5.  **Construct Tool Call:**\n   ```json\n   {\n     "name": "Movie",\n     "parameters": {\n       "title": "Inception",\n       "year": 2010,\n       "director": "Christopher Nolan",\n       "rating": 8.8\n     }\n   }\n   ```\n6.  **Execute Tool Call:** (Simulated) The tool w

### Nested structure

In [24]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    grenres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Miles')], grenres=['Science Fiction', 'Action', 'Thriller', 'Adventure'], budget=160.0)

### TypeDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [45]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "This year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie rating out of 10"]

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
    

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [30]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Avengers")
response
    

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': 2012}

In [34]:
model.profile

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [38]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="Name of the person")
    email: str = Field(description="email of the person")
    phone: str = Field(description="Phone no of the person")

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='391a7bae-cc14-4073-9d64-e08974d57710'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - Input text: "John Doe, john@example.com, (555) 123-4567"\n   - Task: Extract contact info\n   - Available function: `ContactInfo` with parameters `name`, `email`, `phone` (all required)\n\n2.  **Identify Parameters from Input:**\n   - Name: "John Doe"\n   - Email: "john@example.com"\n   - Phone: "(555) 123-4567"\n\n3.  **Map to Function Schema:**\n   - `name`: "John Doe"\n   - `email`: "john@example.com"\n   - `phone`: "(555) 123-4567"\n\n4.  **Construct Function Call:**\n   - Call `ContactInfo` with the extracted values.\n\n5.  **Output Generation:**\n   - Generate the function call in the specified format.✅\n   - Check constraints: All required parameters are pres

In [39]:
result['structured_response']

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [40]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [43]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')